In [ ]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langchain.agents.structured_output import ToolStrategy
from typing import Literal, Dict, Any
from dotenv import load_dotenv
from pydantic import BaseModel, Field, model_validator
load_dotenv()

class ModelResponseFormat(BaseModel):
    """Agent Response format"""
    information: str
    claim: str
    source_quote: str = Field(description="the source information used")


model = ChatGroq(
    model="openai/gpt-oss-20b"
)
model_with_structure = model.with_structured_output(ModelResponseFormat)

@tool
def web_search(query: str, max_results: int = 2, topics: Literal["general", "news", "finance" ] = "general", include_raw_content: bool = True) -> Dict[str, Any]:
    """web search tool that reterives the data from web"""
    response =  TavilySearch(topic=topics, max_results= 1, include_raw_content=False).invoke(query)
    # Format only the essential fields into a compact string
    clean_results = []
    for item in response.get("results", []):
        clean_results.append(
            f"Title: {item.get('title')}\n"
            f"URL: {item.get('url')}\n"
            f"Snippet: {item.get('content')[:500]}"
        )
        
    print("websearch result : ", clean_results)
    return "\n\n---\n\n".join(clean_results)


In [ ]:
model.invoke("what is the remaining token i have")

In [ ]:
"""
Implement Quote Verification
tell the agent to return the claim along with source quote 
compare with reterived document 
if source quote in reterived document 
returns Evidence Found
else return evidence not found
"""

agent = create_agent(
    model=model,
    system_prompt="""
    you are a helpful assistant which fetches the current data using web search tool when asked, 
    only reterive maximum 3 websearch results and relay on it for evidence grounded answer
    and return the response claim along with source claim which you used for generating the respone in provided structure.
    """,
    tools=[web_search],
    response_format=ToolStrategy(ModelResponseFormat)
)
agent

In [29]:
try:
    agent_res = agent.invoke({
        "messages": [{
            "role": "user",
            "content": "According to current Market standards which Car is better in terms of performace and reliability either Volkswagen virtus or Thar rox 4x4"
        }]
    })
except Exception as e:
    print("Status Code:", getattr(e, "status_code", "Unknown"))
    print("Error Body:", getattr(e, "body", str(e)))


websearch result :  ['Title: Volkswagen\nURL: https://tavily.com/knowledge/company/3l5Yk4YZIkCQ\nSnippet: # Volkswagen India\nID: 3l5Yk4YZIkCQ\nWebsite: www.volkswagen.co.in\nLegal name: Volkswagen\nHeadquarters: Mumbai, Maharashtra, India\nLinkedIn: www.linkedin.com/company/volkswagen-india · 493,645 followers\nIndustry: Automotive\nIndustry codes: SIC 55, 551, NAICS 42, 423\nCompany type: Privately Held\nFounded: 2007 [...] Description: Volkswagen India is a company that offers German-engineered cars, including Tiguan R-Line, Golf GTI, Tayron, and 5-star GNCAP rated Virtus & Taigun. They provide a range of']
websearch result :  ['Title: Volkswagen Sedans in India: Sales, Reliability and Quality Compared | Ajay S posted on the topic | LinkedIn\nURL: https://www.linkedin.com/posts/ajay-selvam_volkswagen-sedans-in-india-a-decade-of-activity-7517667810257600512-9ZUa\nSnippet: mistake is to use crash-test performance as proof of mechanical reliability. Crashworthiness measures how well a 

In [44]:
research = agent_res["structured_response"]

print(type(research))
print(repr(research))
for item in research:
    print(type(item), repr(item))

for i, doc in enumerate(retrieved_documents):
    print(f"\nDOCUMENT {i}")
    print(type(doc))
    print(repr(doc)[:1000])

<class '__main__.ModelResponseFormat'>
ModelResponseFormat(information='The Virtus is equipped with a 1.5‑litre TSI petrol engine that can be mated to a 7‑speed DSG or a 6‑speed manual, giving it a more powerful and efficient power‑train for city and highway driving.  The Thar\u202fRox\u202f4×4, on the other hand, is built on a robust chassis, has a 4‑wheel‑drive system, and has been praised for its strong build and consistent performance on varied road conditions, which is a key indicator of long‑term reliability.', claim='According to current market standards, the Volkswagen\u202fVirtus scores higher on straight‑line performance and on‑road refinement, while the Mahindra\u202fThar\u202fRox\u202f4×4 is judged more reliable and better suited for rugged use and long‑term durability.', source_quote='"The build is massive and dependable. Safety is 5-star. It performs very well on all types of roads. ..." (Thar\u202fRox review).  "Volkswagen India … offers German‑engineered cars, including

In [48]:
# Apply quote verification directly in agent response
from langchain_core.messages.tool import ToolMessage
import re

retrieved_documents = []
for message in agent_res['messages']:
    if isinstance(message, ToolMessage):
        retrieved_documents.append(message.content)
result = agent_res['structured_response']
information = result.information
claim = result.claim
source_quote = result.source_quote


print(f"information: {type(information)}")
print(f"claim: {type(claim)}")
print(f"source_quote: {type(source_quote)}")
def quote_verification(source_quote, reterived_documents):
    src_quotes = re.escape(source_quote)
    reterived_docs = "\n\n".join(reterived_documents)
    if re.search(src_quotes, reterived_docs):
        return "Evidence verified"
    return "Evidence not found"

print(quote_verification("The build is massive and dependable", retrieved_documents))

information: <class 'str'>
claim: <class 'str'>
source_quote: <class 'str'>
Evidence verified


In [28]:
information, claim, source_quote = agent_res['structured_response']
print(f"{claim} \n\n {source_quote}")

('claim', 'Based on recent reviews and market data, the Volkswagen\u202fVirtus delivers superior performance and reliability compared to the Mahindra\u202fThar\u202fRox\u202f4×4.') 

 ('source_quote', 'The 2024 Volkswagen Virtus GT Plus ... 1.5\u202fL TSI Turbo‑petrol engine delivers strong performance, the DSG gearbox shifts quickly, and the handling is precise and confidence‑inspiring.\n\nThe Volkswagen Virtus has not failed in India. It remains an active and well‑regarded sedan known for its solid German build quality, punchy turbo‑petrol engines, and 5‑star Global NCAP safety rating.\n\nPower 1997\u202fCC vs 1498\u202fCC; Mileage 12–15\u202fkm/l vs 18–20\u202fkm/l')


In [50]:
research_result = {
    "claim": "LiteLLM supports model routing.",
    "source_quote": "LiteLLM supports routing across multiple LLM providers."
}
retrieved_document = """
LiteLLM provides a unified interface for interacting
with multiple LLM providers.

LiteLLM supports routing across multiple LLM providers.

It also provides logging and usage tracking.
"""

In [51]:
"""
Implement a prototype of quote verification
Quote verification determines whether the sources which agent uses for generating the response are from reterived document or from the actual source
"""

import re 

if re.search(research_result["source_quote"], retrieved_document, re.IGNORECASE):
    print("Evidence verified")
else:
    print("Evidence not found")

Evidence verified
